# Step 12.2 — Causal Mask：怎样让 Future Positions 在 Softmax 后真的变成 0？

Step 12.1 已经得到 autoregressive visibility rule：

$$
\boxed{j\le i}
$$

其中 row `i` 是 Query position，column `j` 是 Key position。

也就是说：

```text
j < i  -> past，允许
j = i  -> current，允许
j > i  -> future，禁止
```

现在要把这条规则真正写进 attention score matrix。

最常见做法是 **additive causal mask（加法式因果掩码）**：允许位置加 0，未来位置加 `-inf`，然后再做 softmax。

本节只讲 causal mask 本身；不进入 Step 12.3 的数值稳定性专题，也不实现完整 attention head。


## 1. 为什么 Future Positions 正好是 Strict Upper Triangle？

对一个 `T×T` score matrix：

```text
rows    = Query positions i
columns = Key positions j
```

未来位置满足：

$$
j>i.
$$

把 row index `i` 纵向、column index `j` 横向排出来，`j>i` 正好位于主对角线之上的 **strict upper triangle（严格上三角）**。

例如 `T=4`：

```text
           Key j
          0   1   2   3
Query 0   ✓   ×   ×   ×
Query 1   ✓   ✓   ×   ×
Query 2   ✓   ✓   ✓   ×
Query 3   ✓   ✓   ✓   ✓
```

主对角线 `j=i` 是允许的，因为当前位置 `x_i` 本身属于预测 `x_(i+1)` 的合法上下文。

因此需要屏蔽的是：

```text
strict upper triangle only
```

而不是包含 diagonal 的整个 upper triangle。


## 2. Additive Mask：允许位置加 0，Future 位置加 -∞

定义 causal mask：

$$
\boxed{
M_{i,j}
=
\begin{cases}
0,&j\le i\\
-\infty,&j>i
\end{cases}
}
$$

对 `T=4`：

$$
M=
\begin{bmatrix}
0&-\infty&-\infty&-\infty\\
0&0&-\infty&-\infty\\
0&0&0&-\infty\\
0&0&0&0
\end{bmatrix}.
$$

如果原本的 scaled compatibility scores 是：

$$
S_{scaled}\in\mathbb R^{B\times T\times T},
$$

那么 masked scores：

$$
\boxed{
S_{masked}
=
S_{scaled}+M
}
$$

其中 `(T,T)` mask 可以通过 broadcasting 作用到每个 batch sample。

允许位置：

$$
s+0=s
$$

不会变化。

未来位置：

$$
s+(-\infty)=-\infty
$$

无论原来的 compatibility score 多大，都被强制变成不可参与 retrieval 的值。


In [ ]:
import mlx.core as mx

T = 4

# mx.triu(..., k=1) keeps only entries strictly above the main diagonal.
future_upper = mx.triu(mx.ones((T, T)), k=1)

# Additive mask: future -> -inf, allowed past/current -> 0.
causal_mask = mx.where(
    future_upper > 0,
    float('-inf'),
    0.0,
)

print('future upper triangle:')
print(future_upper)
print('causal additive mask:')
print(causal_mask)
print('mask shape:', causal_mask.shape)


## 3. 为什么 `-∞` 经过 Softmax 后 Weight 会变成 0？

对一个 Query row，softmax 是：

$$
A_{i,j}
=
\frac{\exp(S_{masked,i,j})}
{\sum_k\exp(S_{masked,i,k})}.
$$

如果 position `j` 是 future position：

$$
S_{masked,i,j}=-\infty.
$$

数学上：

$$
\boxed{
\exp(-\infty)=0
}.
$$

于是 numerator：

$$
\exp(-\infty)=0.
$$

所以：

$$
\boxed{A_{i,j}=0}
$$

只要这一行至少还有一个允许的 finite score，softmax denominator 就由允许位置贡献。

而 causal mask 保留 diagonal，所以每一行至少允许当前 position `j=i`；不会出现“整行全被 mask 掉”的情况。

因此 causal mask 并不是在 softmax 后再把概率手工清零，而是**先改变 logits/scores，再让 softmax 自然产生 0 future weight**。


## 4. 一个三位置小例子：Future Score 再高也必须归零

假设 `T=3`，scaled scores 是：

$$
S_{scaled}
=
\begin{bmatrix}
1&9&8\\
2&1&7\\
1&2&3
\end{bmatrix}.
$$

注意 Query 0 对 future Keys 1 和 2 的 raw compatibility 非常高：9 和 8。

如果没有 causal constraint，Query 0 会强烈读取未来。

加入 mask：

$$
M=
\begin{bmatrix}
0&-\infty&-\infty\\
0&0&-\infty\\
0&0&0
\end{bmatrix}.
$$

于是：

$$
S_{masked}
=
\begin{bmatrix}
1&-\infty&-\infty\\
2&1&-\infty\\
1&2&3
\end{bmatrix}.
$$

### Query 0

只有 Key 0 合法，所以：

$$
A_{0,:}=[1,0,0].
$$

即使原来的 future score 是 9，也完全不能使用。

### Query 1

只在 scores `[2,1]` 上归一化，future Key 2 的 weight 必须是 0。

### Query 2

它已经位于最后一个位置，没有 future Keys，因此三个位置都可以参与 softmax。

这说明 causal masking 控制的是**可见性资格**，而不是“future score 较小时才忽略”。


In [ ]:
scaled_scores = mx.array([
    [1.0, 9.0, 8.0],
    [2.0, 1.0, 7.0],
    [1.0, 2.0, 3.0],
])  # (T,T)

T = scaled_scores.shape[-1]
future_upper = mx.triu(mx.ones((T, T)), k=1)
causal_mask = mx.where(future_upper > 0, float('-inf'), 0.0)

masked_scores = scaled_scores + causal_mask
attention_weights = mx.softmax(masked_scores, axis=-1)

print('scaled_scores:')
print(scaled_scores)
print('masked_scores:')
print(masked_scores)
print('attention_weights:')
print(attention_weights)
print('row sums:', mx.sum(attention_weights, axis=-1))


## 5. Batch Shape 与 Broadcasting：Mask 为什么只需要 `(T,T)`？

真实 Self-Attention scaled scores：

$$
S_{scaled}:(B,T,T).
$$

而 causal visibility rule 只取决于 sequence positions `i,j`，不取决于 batch sample。

所以基础 causal mask 可以只有：

$$
M:(T,T).
$$

相加时 broadcasting：

$$
(B,T,T)
+
(T,T)
\rightarrow
(B,T,T).
$$

然后：

$$
A
=
\operatorname{softmax}(S_{masked},\ axis=-1)
$$

仍然得到：

$$
A:(B,T,T).
$$

每个 sample 使用同一个 causal visibility pattern，但自己的 compatibility scores 仍然完全不同。

### Mask 与 Attention Weight 不是同一个东西

需要区分：

```text
causal mask
→ 描述哪些 Query–Key pairs 具有访问资格

attention weights
→ 在所有合法 positions 中，模型实际分配多少权重
```

Mask 是任务结构约束；Attention weights 则来自当前 Q/K 内容。


## 6. Step 12.2 到这里形成的完整逻辑

Step 12.1 给出规则：

$$
j\le i.
$$

Step 12.2 把规则写成 additive mask：

$$
M_{i,j}
=
\begin{cases}
0,&j\le i\\
-\infty,&j>i.
\end{cases}
$$

然后：

$$
S_{masked}
=
\frac{QK^T}{\sqrt{d_k}}+M.
$$

再：

$$
\boxed{
A
=
\operatorname{softmax}(S_{masked},\ axis=-1)
}
$$

于是所有 future positions 都得到：

$$
\boxed{A_{i,j}=0\quad\text{for }j>i}.
$$

这就是从 general self-attention 到 causal self-attention 最关键的 visibility mechanism。

### 本节边界

- strict upper triangle 对应 `j>i` 的 future positions；diagonal `j=i` 保留。
- additive causal mask 在合法位置放 0，在未来位置放 `-inf`。
- `scaled_scores + mask` 不改变 `(B,T,T)` shape；`(T,T)` mask 可广播到 batch。
- 因为 `exp(-inf)=0`，softmax 后 future attention weights 为 0。
- causal mask 决定“能不能看”，attention weights 决定“合法位置里看多少”。
- 本轮只准备 Step 12.2；不进入 Step 12.3，不展开 stable-softmax / numerical-stability 专题，也不实现完整 causal attention head。
